In [2]:
%load_ext autoreload
%autoreload 2

import os
import sys
import requests
import pandas as pd
import datetime
import time
import re
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
load_dotenv("../../config/.env") 
contact_email = os.getenv("CONTACT_EMAIL", "anonymous@example.com")

notebook_dir = os.path.dirname(os.path.abspath("__file__"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
os.makedirs(raw_data_dir, exist_ok=True)

# Import Schema Manager
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in the config directory.")

# --- 2. Registry Lookup & Target Setup ---
SOURCE_PREFIX = "LCDGT"

registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir, 
    fallback_name="LC Demographic Group Terms",
    fallback_uri="http://id.loc.gov/authorities/demographicTerms/"
)

SOURCE_NAME = registry_data['Source_Name']
BASE_URI = registry_data['Base_URI']

raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")
COLLECTION_URL = "https://id.loc.gov/search/?q=memberOf:http://id.loc.gov/authorities/demographicTerms/collection_LCDGT_Religion&fo=json&count=500"

HEADERS = {
    'User-Agent': f'ReligiousMappingProject/1.0 (Research; mailto:{contact_email})',
    'Accept': 'application/json'
}

label_cache = {}
member_uris = set()

# --- 3. Helper Functions ---
def get_label_only(uri):
    """Fetches just the English preferred label for building hierarchy paths."""
    if not uri: return ""
    clean_u = uri.rstrip('.json').rstrip('.rdf').rstrip('/')
    if clean_u in label_cache: return label_cache[clean_u]
    try:
        res = requests.get(f"{clean_u}.json", headers=HEADERS, timeout=10)
        if res.status_code == 200:
            data = res.json()
            if isinstance(data, dict): data = [data]
            node = next((item for item in data if isinstance(item, dict) and item.get('@id') == clean_u), {})
            
            # Extract English label
            for l_node in node.get('http://www.w3.org/2004/02/skos/core#prefLabel', []):
                lang = l_node.get('@language', '').lower()
                if not lang or lang.startswith('en'):
                    label = l_node.get('@value', '')
                    label_cache[clean_u] = label
                    return label
    except: pass
    return clean_u.split('/')[-1]

def get_lc_details(uri):
    """Fetches full metadata for an LC demographic term, checking for translations."""
    clean_uri = uri.rstrip('/').replace('.json', '').replace('.rdf', '').replace('.html', '')
    
    for attempt in range(3):
        try:
            res = requests.get(f"{clean_uri}.json", headers=HEADERS, timeout=15)
            if res.status_code != 200: 
                time.sleep(1)
                continue
            
            data = res.json()
            if isinstance(data, dict): data = [data]
            node = next((item for item in data if isinstance(item, dict) and item.get('@id') == clean_uri), {})
            
            SKOS_PREF = 'http://www.w3.org/2004/02/skos/core#prefLabel'
            SKOS_ALT  = 'http://www.w3.org/2004/02/skos/core#altLabel'
            SKOS_BROADER = 'http://www.w3.org/2004/02/skos/core#broader'
            MADS_CITATION = 'http://www.loc.gov/mads/rdf/v1#citationNote'

            # Extract Label & Detect Translations
            label = "No Label"
            has_translation = ""
            synonyms = []
            
            # Check Preferred Labels
            for pref in node.get(SKOS_PREF, []):
                lang = pref.get('@language', '').lower()
                if not lang or lang.startswith('en'):
                    label = pref.get('@value', label)
                else:
                    has_translation = "yes"
            
            label_cache[clean_uri] = label
            
            # Check Alt Labels (Synonyms)
            for alt in node.get(SKOS_ALT, []):
                lang = alt.get('@language', '').lower()
                if not lang or lang.startswith('en'):
                    if alt.get('@value'): synonyms.append(alt.get('@value'))
                else:
                    has_translation = "yes"
            
            # Extract Descriptions
            descriptions = []
            for item in data:
                if isinstance(item, dict):
                    for n in item.get(MADS_CITATION, []):
                        if n.get('@value'): descriptions.append(n.get('@value'))
            
            # Extract Parents
            broader_nodes = node.get(SKOS_BROADER, [])
            p_ids = [b.get('@id').split('/')[-1] for b in broader_nodes if b.get('@id')]
            p_labels = [get_label_only(b.get('@id')) for b in broader_nodes if b.get('@id')]

            return {
                'label': label,
                'synonyms': " | ".join(list(set(synonyms))),
                'description': " | ".join(descriptions),
                'parent_ids': " | ".join(p_ids),
                'parent_labels': " | ".join(p_labels),
                'has_translation': has_translation
            }
        except Exception:
            time.sleep(2) # Back off and try again
    return None

# --- 4. Discovery Phase ---
print(f"Discovering Collection for {SOURCE_PREFIX}...")

try:
    response = requests.get(COLLECTION_URL, headers=HEADERS, timeout=20)
    if response.status_code == 200:
        search_data = response.json()
        
        def extract_uris(obj):
            if isinstance(obj, dict):
                for k, v in obj.items():
                    if isinstance(v, str) and '/demographicTerms/dg' in v:
                        match = re.search(r'(dg\d+)', v)
                        if match: 
                            member_uris.add(f"{BASE_URI}{match.group(1)}")
                    extract_uris(v)
            elif isinstance(obj, list):
                for i in obj: 
                    extract_uris(i)

        extract_uris(search_data)
        print(f"Discovery complete. Found {len(member_uris)} URIs.")
    else:
        print(f"Search failed. HTTP Status: {response.status_code}")
except Exception as e:
    print(f"Discovery error: {e}")

# --- 5. Extraction Phase ---
total = len(member_uris)
if total > 0:
    print(f"\nStarting ingest for {total} terms...")
    if os.path.exists(raw_ingest_file): os.remove(raw_ingest_file)
    
    for i, uri in enumerate(sorted(list(member_uris)), 1):
        cid = uri.split('/')[-1]
        
        meta = get_lc_details(uri)
        if not meta: continue
            
        path = f"{meta['parent_labels']} > {meta['label']}" if meta['parent_labels'] else meta['label']
        
        display_text = f"[{i}/{total}] Ingesting: {meta['label'][:60]}"
        print(f"\r{display_text}{' ' * 40}", end="", flush=True)

        # Pack raw data
        extracted_data = {
            'Source_System': SOURCE_NAME, 
            'Primary_Label': meta['label'],
            'CURIE': f"{SOURCE_PREFIX}:{cid}", 
            'Formal_Label': meta['label'],
            'Concept_Type': 'skos:Concept',
            'Hierarchy_Path': path, 
            'Synonyms': meta['synonyms'],
            'Description': meta['description'], 
            'Parent_IDs': meta['parent_ids'],
            'Concept_ID': cid, 
            'URI': uri, 
            'Has_Translation': meta['has_translation'],
            'Status': 'active'
        }
        
        # MAGIC HAPPENS HERE: Validate and format to 14 columns
        clean_row = finalize_row(extracted_data)
        
        df_row = pd.DataFrame([clean_row])[COLUMN_ORDER]
        file_exists = os.path.isfile(raw_ingest_file)
        df_row.to_csv(raw_ingest_file, mode='a', index=False, header=not file_exists, encoding='utf-8-sig')
            
        time.sleep(0.5) 
else:
    print("\nNo URIs found to ingest.")

print(f"\n\nIngest Complete! Saved to {raw_ingest_file}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Discovering Collection for LCDGT...
Discovery complete. Found 95 URIs.

Starting ingest for 95 terms...
[95/95] Ingesting: Persons...                                                                          

Ingest Complete! Saved to c:\Users\njsgi\Documents\GitHub\religion-ontology-mapping\data\raw\raw_lcdgt.csv
